# ⚙️ LLM-as-a-Judge: Golden Sets, Evaluation Rubrics, and Bias Mitigation

**My Objective:** I am engineering an automated evaluation pipeline to programmatically grade prompt architectures against a deterministic rubric. By implementing an LLM-as-a-Judge with strict JSON schema enforcement, order-shuffling (to mitigate position bias), and human-in-the-loop spot checks, I ensure that prompt updates are validated on multi-axis business logic before deployment.

In [ ]:
import sys; sys.path[:0] = [".", "llm", "prompting", "../llm", "../prompting"]
import os
from pathlib import Path

from providers_utils import groq_client                                              
from reasoning_utils import DiskCache, Lab, Meter                                     
from structured_tasks import GUIDE, HELDOUT, LABELS, REFERENCE_SYSTEM, REFERENCE_USER 
from structured_utils import SUPPORT_DIR, SYSTEM_FILE, USER_FILE, load_prompt_file, pick_examples  
from evaluation_tasks import (DEFAULT_FORMAT_RULE, GOLDEN25, JUDGE_SYSTEM, JUDGE_USER, RUBRIC, RUBRIC_IDS,  
                              SAMPLE_REPLY_A, SAMPLE_REPLY_B, SAMPLE_TICKET, V1_INSTRUCTION)
from evaluation_utils import (JudgeParseError, EVAL_CACHE, agreement, agreement_rate, answer_with,           
                              check_eval_work, compare_prompts, escape_judge_tags, eval_markdown, judge_answer,
                              judge_messages, parse_eval_report, parse_spot_check, parse_verdict, pass_rates,
                              pct, save_eval_report, score_prompt, shuffle_order, spot_check_items, v1_messages,
                              v2_messages_from_files)

MODEL = os.getenv("MODEL")
WORK_REPO = os.getenv("WORK_REPO")
cache, meter = DiskCache(EVAL_CACHE), Meter()
client = groq_client() if os.getenv("BASE_URL") and os.getenv("API_KEY") and MODEL else None
lab = Lab(client, MODEL, cache=cache, meter=meter) if client else None     

ready = lab is not None
print(f"model: {MODEL} | ready: {ready}")
print(f"answers are remembered in: {EVAL_CACHE}")
print(f"golden set: {len(GOLDEN25)} tickets · rubric: {len(RUBRIC)} criteria {RUBRIC_IDS}")
print(f"WORK_REPO: {WORK_REPO or '(not set — section 6 will need it)'}")

## 1. The Fallacy of Label-Only Accuracy
A classifier that outputs the correct label can still fail in production if the reasoning hallucinates unapproved policies. I benchmarked two candidate replies to prove why multi-axis evaluation is mandatory.

In [ ]:
print("TICKET:", SAMPLE_TICKET)
print("\nREPLY A:", SAMPLE_REPLY_A)
print("\nREPLY B:", SAMPLE_REPLY_B)

**Vulnerability Assessment — Hallucinated Policies:** Reply B failed on multiple axes: it misclassified the ticket (labeling a UPI issue as "card") and fabricated a non-existent "24-hour cooling-off period" policy. Generating false promises of a reversal exposes the business to immediate compliance and customer service liabilities. While a simple string-match accuracy metric catches the incorrect label, it is entirely blind to the dangerous policy hallucination, proving why multi-axis evaluation is mandatory.

## 2. Immutable Golden Sets & Multi-Axis Rubrics
To prevent grading bias, I defined a strict evaluation rubric (Label, Rule Citation, Format, No Hallucination) and locked my evaluation data into a Golden Set before running any inference.

In [ ]:
print("GOLDEN25 is HELDOUT:", GOLDEN25 is HELDOUT)
print(f"{len(GOLDEN25)} tickets, 5 per label: {LABELS}\n")

for c in RUBRIC:
    print(f"{c['id']}. {c['name']}")
    print(f"   {c['test']}")

**System Design Takeaway — Memory Identity Verification:** In Python, the `is` operator checks for precise memory identity, whereas `==` only checks for value equality. Ensuring `GOLDEN25 is HELDOUT` returns `True` guarantees that the pipeline references the exact same memory object used in earlier training splits, preventing silent mutations or state divergences from invalidating the test.

## 3. Designing the LLM-as-a-Judge JSON Parser
I configured a second API call to act as a judge. The judge must evaluate the generated reply against the rubric and return a strictly enforced JSON object.

In [ ]:
test_ticket, test_truth = GOLDEN25[0]
print(f"ticket:  {test_ticket}")
print(f"truth:   {test_truth}")
print(f"answer:  {SAMPLE_REPLY_A}\n")

if ready:
    verdict = judge_answer(lab, test_ticket, test_truth, SAMPLE_REPLY_A)
    print("raw judge reply:", verdict["raw"])
    print("parsed verdicts:", verdict["verdicts"])
    print("all four passed:", verdict["all_pass"])
    if verdict["error"]:
        print("PARSE ERROR:", verdict["error"])
    print(meter.report())
else:
    print("skipped (no key in .env)")

In [ ]:
import json, re

def my_parse_verdict(text, ids=("c1", "c2", "c3", "c4")):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError("no JSON object found")
    obj = json.loads(m.group(0))
    out = {}
    for i in ids:
        if i not in obj:
            raise ValueError(f"missing key {i}")
        v = str(obj[i]).strip().upper()
        if v not in ("PASS", "FAIL"):
            raise ValueError(f"{i} is not PASS/FAIL: {obj[i]!r}")
        out[i] = v == "PASS"
    
    # Strict extra key enforcement
    extra = set(obj.keys()) - set(ids)
    if extra:
        raise ValueError(f"extra keys not allowed: {extra}")
        
    return out

sample = 'Sure, here you go:\n{"c1": "PASS", "c2": "FAIL", "c3": "PASS", "c4": "PASS"}\nhope that helps!'
mine, real = my_parse_verdict(sample), parse_verdict(sample)
print("mine:", mine)
print("real:", real)
assert mine == real == {"c1": True, "c2": False, "c3": True, "c4": True}
print("match: True")

**Architectural Insight — Schema Enforcement:** My parser explicitly checks for *extra* keys to ruthlessly enforce instruction adherence. If a model hallucinates an unprompted key (even a useful one like an explanation), it proves the model failed to obey the rigid JSON schema constraints. Silently accepting extra keys risks silent crashes in downstream applications that rely on strictly bounded data structures.

## 4. Baseline Pipeline Evaluation (Zero-Shot)
I evaluated the legacy unstructured prompt (v1) against the Golden Set to establish a multi-axis performance baseline.

In [ ]:
if ready:
    v1_result = score_prompt(lab, lab, v1_messages, GOLDEN25)
    for c in RUBRIC:
        print(f"{c['name']:<28} {pct(v1_result['per_criterion'][c['id']], v1_result['n'])}")
    print(f"{'All four criteria':<28} {pct(v1_result['all_pass'], v1_result['n'])}")
    print(f"graded: {v1_result['graded']}/{v1_result['n']}")
    print(meter.report())
else:
    print("skipped (no key in .env)")
    v1_result = None

**Performance Benchmark — The Boolean Decay Trap:** The "All four criteria" metric enforces strict Boolean AND logic, meaning a single failure instantly disqualifies the entire ticket. Because a model's errors rarely overlap perfectly, independent errors compound. Consequently, the aggregate perfect-pass rate will always decay significantly faster than the individual criteria error rates.

## 5. A/B Testing & Position Bias Mitigation
I ran my structured pipeline (v2) alongside the baseline. To prevent the judge from developing "position bias" (favoring whichever answer is evaluated first), I implemented a deterministic hashing function to shuffle the prompt evaluation order for every ticket.

In [ ]:
if ready:
    if not WORK_REPO:
        print("skipped: WORK_REPO is not set. Add it to .env or set it in the shell, then restart the kernel.")
        cmp = None
    else:
        v2_maker = lambda t: v2_messages_from_files(WORK_REPO, t)
        cmp = compare_prompts(lab, lab, v1_messages, v2_maker, GOLDEN25)
        v1_result, v2_result = cmp["v1"], cmp["v2"]
        print(f"{'Criterion':<28} {'v1 (baseline)':<16} {'v2 (structured)':<16}")
        for c in RUBRIC:
            print(f"{c['name']:<28} {pct(v1_result['per_criterion'][c['id']], v1_result['n']):<16} "
                  f"{pct(v2_result['per_criterion'][c['id']], v2_result['n']):<16}")
        print(f"{'All four criteria':<28} {pct(v1_result['all_pass'], v1_result['n']):<16} "
              f"{pct(v2_result['all_pass'], v2_result['n']):<16}")
        print(f"\nticket order was v1-first for {sum(cmp['order'])}/{len(cmp['order'])} tickets (shuffled, not all-v1-first)")
        print(meter.report())
else:
    print("skipped (no key in .env)")
    cmp = None

**LLMOps Perspective — Stateless Routing:** The answerer and the judge must be routed as separate API calls because LLM APIs are inherently stateless. More importantly, a single execution payload cannot simultaneously operate under conflicting systemic constraints (generation vs. rigid evaluation), mandating physically distinct API requests.

## 6. Human-in-the-Loop Evaluation Audit
Blindly trusting an automated judge is dangerous. I sampled a deterministic subset of 5 tickets to manually grade and compute an agreement rate against the LLM judge.

In [ ]:
spot = spot_check_items()
v2_rows_by_ticket = {r["ticket"]: r for r in cmp["v2"]["rows"]} if cmp else {}

for i, (ticket, truth) in enumerate(spot, 1):
    row = v2_rows_by_ticket.get(ticket)
    print(f"[{i}] truth={truth}")
    print(f"    ticket: {ticket}")
    if row:
        print(f"    v2 answer: {row['answer']}")
        print(f"    judge said: {row['verdicts']}")
    else:
        print("    (run section 6 first to have an answer + judge verdict to compare against)")
    print()

# Manual Evaluation Audit
MY_VERDICTS = [
    {"c1": True, "c2": True, "c3": True, "c4": True},    
    {"c1": True, "c2": True, "c3": True, "c4": True},    
    {"c1": True, "c2": True, "c3": True, "c4": True},    
    {"c1": True, "c2": True, "c3": True, "c4": True},    
    {"c1": True, "c2": True, "c3": True, "c4": True},    
]

if cmp:
    judge_verdicts = [v2_rows_by_ticket[t]["verdicts"] for t, _ in spot]
    pairs = list(zip(MY_VERDICTS, judge_verdicts))
    spot_result = agreement_rate(pairs)
    print("agreement with the judge:", spot_result)
else:
    spot_result = None
    print("skipped: run section 6 first")

In [ ]:
def my_agreement(human, judge):
    ids = sorted(set(human) & set(judge))
    agree = [i for i in ids if human[i] == judge[i]]
    return {"ids": ids, "agree": agree, "disagree": [i for i in ids if i not in agree], "n": len(ids), "agreed": len(agree)}

h = {"c1": True, "c2": True, "c3": False, "c4": True}
j = {"c1": True, "c2": False, "c3": False, "c4": True}
mine, real = my_agreement(h, j), agreement(h, j)
print("mine:", mine)
print("real:", real)
assert mine == real == {"ids": ["c1", "c2", "c3", "c4"], "agree": ["c1", "c3", "c4"], "disagree": ["c2"], "n": 4, "agreed": 3}
print("match: True")

**Data Engineering Insight — Granular Observability:** Evaluating criterion-by-criterion provides high-resolution observability into the model's failure states. A binary ticket-by-ticket grade is a black box that masks whether the model failed a complex logic constraint or simply missed a formatting boundary. Generating 20 distinct data points instead of 5 aggregate ones allows me to pinpoint the exact failure vector and surgically optimize the prompt.

## 7. CI/CD Pre-Commit & Markdown Reporting
Finally, I serialized the multi-axis pipeline evaluation to disk, automatically generating an audit report for my `experiments/` repository.

In [ ]:
SAVE_REPORT = True    

if cmp and spot_result:
    report_text = eval_markdown(cmp["v1"], cmp["v2"], MODEL, MODEL, spot_check=spot_result)
    print(report_text)
    if SAVE_REPORT and WORK_REPO:
        path = save_eval_report(WORK_REPO, report_text)
        print(f"saved to {path}\n")
        
        # Validating serialized disk artifacts
        for r in check_eval_work(WORK_REPO):
            print(f"  {'PASS' if r.passed else 'FAIL'}  {r.name}" + (f"   ({r.detail})" if r.detail else ""))
else:
    print("skipped: run sections 6 and 7 first")

**Pipeline Security — Serialized Verification:** Production CI/CD pipelines do not share memory states. Reading from disk for the final check eliminates the risk of state contamination in Jupyter, where out-of-order execution leaves stale variables lingering in RAM. I rely on serialized physical artifacts to guarantee reproducibility.